In [ ]:
# CELL 1 — SETUP, PROJECT CONFIGURATION & INPUT DISCOVERY
from pathlib import Path
import os, sys, subprocess, importlib.util, zipfile, io, re, json, sqlite3, textwrap, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
PROJECT = ROOT / 'Project_11_Cybersecurity_NSLKDD_SOC_Analytics'
OUTPUTS = PROJECT / 'outputs'
FIGURES = PROJECT / 'figures'
for p in [PROJECT, OUTPUTS, FIGURES]: p.mkdir(parents=True, exist_ok=True)
SOURCE_URL = 'https://www.kaggle.com/datasets/hassan06/nslkdd'
REPO_NAME = 'Cybersecurity-Network-Intrusion-SOC-Analytics'
COLUMNS = ['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations','num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','attack_type','difficulty']
assert len(COLUMNS)==43
print('Project 11 | NSL-KDD defensive intrusion detection analytics')
print('Input source:',SOURCE_URL)
print('Data caveat: historical academic benchmark; records have no event timestamps and are NOT live SOC events.')
print('Expected inputs: KDDTrain+.txt and KDDTest+.txt OR ZIP containing these files.')
print('Working folder:',PROJECT)

def discover():
    bases=[ROOT,ROOT/'sample_data',Path.cwd()]
    found=[]
    for b in bases:
        if b.exists():
            for pattern in ('*.txt','*.TXT','*.csv','*.zip','*.ZIP'):
                found += list(b.glob(pattern))
    return sorted(set(found))

candidates=discover()
if not any('KDDTrain+' in p.name or 'KDDTest+' in p.name or p.suffix.lower()=='.zip' for p in candidates):
    try:
        from google.colab import files
        print('Upload both KDDTrain+.txt and KDDTest+.txt, or the original Kaggle ZIP:')
        files.upload()
    except ImportError:
        print('Not running inside Google Colab. Put source files next to this notebook.')
print('Discovered:',[p.name for p in discover()])

In [ ]:
# CELL 2 — READ ORIGINAL DATA, VERIFY SCHEMA & AUDIT RAW QUALITY

def read_source(role):
    base_name = 'KDDTrain+' if role=='train' else 'KDDTest+'
    # Prefer full files, not the 20Percent or Test-21 sample.
    wanted=[base_name+'.txt',base_name+'.csv',base_name]
    files=discover()
    for w in wanted:
        for p in files:
            if p.name.lower()==w.lower():
                print(f'{role.upper()}: {p.name}')
                return pd.read_csv(p,header=None,low_memory=False)
    for p in files:
        if p.suffix.lower()=='.zip':
            try:
                with zipfile.ZipFile(p) as z:
                    matching=[name for name in z.namelist() if Path(name).name.lower() in [x.lower() for x in wanted]]
                    if matching:
                        name=matching[0]
                        print(f'{role.upper()}: {p.name} > {name}')
                        with z.open(name) as f: return pd.read_csv(f,header=None,low_memory=False)
            except zipfile.BadZipFile: pass
    raise FileNotFoundError(f'Missing {base_name}.txt. Upload the FULL KDDTrain+.txt and KDDTest+.txt, or Kaggle ZIP, then rerun Cells 1–2.')

train=read_source('train'); test=read_source('test')
for name,frame in [('train',train),('test',test)]:
    if frame.shape[1]!=43: raise ValueError(f'{name} has {frame.shape[1]} columns, expected 43. Check that this is full NSL-KDD KDDTrain+/KDDTest+ without extra headers.')
    frame.columns=COLUMNS
    frame['attack_type']=frame['attack_type'].astype(str).str.strip().str.lower().str.rstrip('.')
    for col in ['protocol_type','service','flag']:
        frame[col]=frame[col].astype(str).str.strip().str.lower()
    for col in [x for x in COLUMNS if x not in ['protocol_type','service','flag','attack_type']]:
        frame[col]=pd.to_numeric(frame[col],errors='coerce')
    frame['split']=name
    frame['is_attack']=(frame['attack_type']!='normal').astype(int)
assert set(train['is_attack'].dropna().unique()).issubset({0,1})
audit=pd.DataFrame([{'split':name,'records':len(df),'columns_original':43,'missing_cells':int(df[COLUMNS].isna().sum().sum()),'duplicate_rows':int(df[COLUMNS].duplicated().sum()),'normal_records':int((df.is_attack==0).sum()),'attack_records':int((df.is_attack==1).sum()),'distinct_attack_labels':int(df.attack_type.nunique())} for name,df in [('train',train),('test',test)]])
audit.to_csv(OUTPUTS/'data_quality_audit.csv',index=False)
print(audit.to_string(index=False))
print('Training labels:',train.attack_type.value_counts().head(12).to_dict())
print('Test labels:',test.attack_type.value_counts().head(12).to_dict())
if train[COLUMNS].isna().any().any() or test[COLUMNS].isna().any().any():
    print('WARNING: Null/invalid numeric values detected; model preparation will impute them. See data_quality_audit.csv.')

In [ ]:
# CELL 3 — DEFENSIVE ATTACK TAXONOMY & SQL ANALYTICS
DOS={'back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb'}
PROBE={'satan','ipsweep','nmap','portsweep','mscan','saint'}
R2L={'guess_passwd','ftp_write','imap','phf','multihop','warezmaster','warezclient','spy','xlock','xsnoop','snmpguess','snmpgetattack','httptunnel','sendmail','named'}
U2R={'buffer_overflow','loadmodule','rootkit','perl','sqlattack','xterm','ps'}
def family(label):
    if label=='normal': return 'Normal'
    if label in DOS: return 'DoS'
    if label in PROBE: return 'Probe'
    if label in R2L: return 'R2L'
    if label in U2R: return 'U2R'
    return 'Other / unmapped'
for df in (train,test): df['attack_family']=df.attack_type.map(family)
combined=pd.concat([train,test],ignore_index=True)
conn=sqlite3.connect(':memory:')
combined[['split','protocol_type','service','flag','attack_type','attack_family','is_attack','src_bytes','dst_bytes','duration','difficulty']].to_sql('connections',conn,index=False,if_exists='replace')
family_sql="""SELECT split,attack_family,COUNT(*) AS records,ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (PARTITION BY split),2) AS share_pct FROM connections GROUP BY split,attack_family ORDER BY split,records DESC"""
family_counts=pd.read_sql_query(family_sql,conn)
protocol_counts=pd.read_sql_query("""SELECT split,protocol_type,COUNT(*) AS records,SUM(is_attack) AS attacks,ROUND(100.0*SUM(is_attack)/COUNT(*),2) AS attack_share_pct FROM connections GROUP BY split,protocol_type ORDER BY split,records DESC""",conn)
service_counts=pd.read_sql_query("""SELECT service,COUNT(*) AS records,SUM(is_attack) AS attacks,ROUND(100.0*SUM(is_attack)/COUNT(*),2) AS attack_share_pct FROM connections WHERE split='test' GROUP BY service HAVING COUNT(*)>=30 ORDER BY attacks DESC LIMIT 15""",conn)
flag_counts=pd.read_sql_query("""SELECT flag,COUNT(*) AS records,SUM(is_attack) AS attacks,ROUND(100.0*SUM(is_attack)/COUNT(*),2) AS attack_share_pct FROM connections WHERE split='test' GROUP BY flag ORDER BY records DESC""",conn)
for name,df in [('attack_family_by_split',family_counts),('protocol_by_split',protocol_counts),('test_service_attack_ranking',service_counts),('test_flag_summary',flag_counts)]: df.to_csv(OUTPUTS/f'{name}.csv',index=False)
print('Family breakdown:');print(family_counts.to_string(index=False))
print('Top tested services:');print(service_counts.head(8).to_string(index=False))
unknown=combined[combined.attack_family=='Other / unmapped'].attack_type.value_counts()
if len(unknown): print('Unmapped labels—review taxonomy:',unknown.to_dict())
print('Note: percentages describe NSL-KDD BENCHMARK SAMPLE composition, NOT real-world intrusion prevalence.')

In [ ]:
# CELL 4 — TEST-ONLY NOVEL ATTACKS, PRIORITY HEURISTICS & COVERAGE LIMITATIONS
train_labels=set(train.attack_type.unique());test_labels=set(test.attack_type.unique())
unseen_labels=sorted(test_labels-train_labels)
unseen=test[test.attack_type.isin(unseen_labels)]
novelty=pd.DataFrame([{'attack_type':lab,'test_records':int((test.attack_type==lab).sum()),'attack_family':family(lab),'seen_in_training':lab in train_labels} for lab in sorted(test_labels)])
novelty.to_csv(OUTPUTS/'test_attack_label_coverage.csv',index=False)
# This is a triage illustration, not a production SOC severity score.
priority_map={'U2R':'High (illustrative)','R2L':'High (illustrative)','DoS':'Medium (illustrative)','Probe':'Review (illustrative)','Normal':'Not an alert','Other / unmapped':'Review (illustrative)'}
priority=test.groupby('attack_family',as_index=False).agg(records=('attack_type','size')).sort_values('records',ascending=False)
priority['suggested_review_label']=priority.attack_family.map(priority_map)
priority.to_csv(OUTPUTS/'illustrative_triage_categories.csv',index=False)
coverage=pd.DataFrame([
 {'dimension':'Event timestamps','available':'No','implication':'Cannot make hourly/daily trends or time-to-detect claims.'},
 {'dimension':'Source/destination IPs','available':'No','implication':'Cannot identify affected real endpoints, geolocations or victims.'},
 {'dimension':'Deployment traffic','available':'No','implication':'No production alert rates or current SOC workload can be inferred.'},
 {'dimension':'Dataset period','available':'Historical academic benchmark','implication':'Do not extrapolate to modern attack prevalence.'},
 {'dimension':'Novel attack labels in test','available':f'{len(unseen_labels)} distinct labels','implication':'Previously unseen labels test generalization but not current threats.'},
])
coverage.to_csv(OUTPUTS/'limitations_and_evidence.csv',index=False)
print('Novel attack labels appearing only in held-out test:',unseen_labels)
print('Novel-label test records:',len(unseen),'of',len(test))
print(priority.to_string(index=False))

In [ ]:
# CELL 5 — HOLDOUT INTRUSION DETECTION BASELINE (NO LABEL LEAKAGE)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix,classification_report,balanced_accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
# Train on canonical training partition, evaluate once on untouched canonical test partition.
# Never use attack_type, is_attack, attack_family, split, or difficulty as model features.
feature_cols=COLUMNS[:41]
cat_cols=['protocol_type','service','flag'];num_cols=[c for c in feature_cols if c not in cat_cols]
Xtr_num=train[num_cols].replace([np.inf,-np.inf],np.nan)
Xte_num=test[num_cols].replace([np.inf,-np.inf],np.nan)
num_imputer=SimpleImputer(strategy='median')
Xtr_n=num_imputer.fit_transform(Xtr_num);Xte_n=num_imputer.transform(Xte_num)
encoder=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
Xtr_c=encoder.fit_transform(train[cat_cols]);Xte_c=encoder.transform(test[cat_cols])
Xtr=np.column_stack([Xtr_n,Xtr_c]).astype(np.float32)
Xte=np.column_stack([Xte_n,Xte_c]).astype(np.float32)
ytr=train['is_attack'].to_numpy(); yte=test['is_attack'].to_numpy()
model=RandomForestClassifier(n_estimators=90,max_depth=18,min_samples_leaf=3,n_jobs=-1,random_state=42,class_weight='balanced_subsample')
model.fit(Xtr,ytr)
pred=model.predict(Xte)
cm=confusion_matrix(yte,pred,labels=[0,1]);tn,fp,fn,tp=cm.ravel()
metrics={'train_records':int(len(train)),'test_records':int(len(test)),'accuracy':float(accuracy_score(yte,pred)),'balanced_accuracy':float(balanced_accuracy_score(yte,pred)),'attack_precision':float(precision_score(yte,pred,zero_division=0)),'attack_recall':float(recall_score(yte,pred,zero_division=0)),'attack_f1':float(f1_score(yte,pred,zero_division=0)),'true_negative':int(tn),'false_positive':int(fp),'false_negative':int(fn),'true_positive':int(tp),'false_positive_rate':float(fp/(fp+tn)) if (fp+tn)>0 else None,'miss_rate':float(fn/(fn+tp)) if (fn+tp)>0 else None,'training_only_categories_encoded_unknown_in_test':True,'seed':42,'model':'RandomForestClassifier','model_scope':'NSL-KDD historical benchmark only'}
(OUTPUTS/'holdout_model_metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
pd.DataFrame([{'actual':'normal' if actual==0 else 'attack','predicted':'normal' if p==0 else 'attack','attack_family':family,'attack_type':lab} for actual,p,family,lab in zip(yte,pred,test.attack_family,test.attack_type)]).groupby(['actual','predicted','attack_family','attack_type'],as_index=False).size().rename(columns={'size':'records'}).to_csv(OUTPUTS/'holdout_error_breakdown.csv',index=False)
importance=pd.DataFrame({'feature':num_cols+cat_cols,'importance':model.feature_importances_}).sort_values('importance',ascending=False)
importance.to_csv(OUTPUTS/'feature_importance.csv',index=False)
print('Held-out TEST metrics:',json.dumps(metrics,indent=2))
print(classification_report(yte,pred,target_names=['normal','attack'],zero_division=0))
print('Caution: high benchmark scores are NOT proof of production performance or modern threat detection.')

In [ ]:
# CELL 6 — EXECUTIVE SOC-STYLE DASHBOARD (HISTORICAL BENCHMARK)
plt.rcParams.update({'figure.dpi':125,'savefig.dpi':160,'font.size':10})
fig,axes=plt.subplots(2,2,figsize=(15,10),constrained_layout=True)
colors=['#0D9488','#334155','#F59E0B','#6366F1','#EF4444','#94A3B8']
order=['Normal','DoS','Probe','R2L','U2R','Other / unmapped']
pivot=family_counts.pivot(index='attack_family',columns='split',values='records').reindex(order).fillna(0)
pivot.plot.bar(ax=axes[0,0],color=['#0D9488','#334155'],rot=30)
axes[0,0].set_title('Family composition by canonical split');axes[0,0].set_xlabel('');axes[0,0].set_ylabel('Records')
top=service_counts.sort_values('attacks',ascending=False).head(10).sort_values('attacks')
axes[0,1].barh(top.service.astype(str),top.attacks,color='#0D9488');axes[0,1].set_title('Test records labelled attack by service');axes[0,1].set_xlabel('Benchmark records')
axes[1,0].imshow(cm,cmap='Blues');axes[1,0].set_title('Holdout confusion matrix');axes[1,0].set_xticks([0,1],['Pred normal','Pred attack']);axes[1,0].set_yticks([0,1],['Actual normal','Actual attack']);
for (r,c),val in np.ndenumerate(cm):axes[1,0].text(c,r,f'{val:,}',ha='center',va='center',fontsize=15,color='black')
axes[1,1].barh(importance.head(10).feature.iloc[::-1],importance.head(10).importance.iloc[::-1],color='#334155');axes[1,1].set_title('Baseline model feature importance');axes[1,1].set_xlabel('RF relative importance')
fig.suptitle('PROJECT 11 | DEFENSIVE NETWORK INTRUSION & SOC ANALYTICS',fontsize=16,weight='bold')
fig.text(.5,-.005,'Historical NSL-KDD train/test partitions | No timestamps, real-time monitoring or production claims',ha='center',fontsize=9)
png=FIGURES/'soc_executive_dashboard.png';fig.savefig(png,bbox_inches='tight',facecolor='white');plt.show();plt.close(fig)
print('Dashboard saved:',png)
# Mechanical totals audit, including partition preservation.
assert sum(family_counts[family_counts.split=='train'].records)==len(train)
assert sum(family_counts[family_counts.split=='test'].records)==len(test)
assert int(cm.sum())==len(test)
assert (tn+fp)==int((test.is_attack==0).sum())
assert (tp+fn)==int((test.is_attack==1).sum())
print('PASS: dashboard input totals and confusion matrix reconcile to source partitions.')

In [ ]:
# CELL 7 — SEVEN EVIDENCE-BASED RECOMMENDATIONS, README & PROJECT FILES
recommendations=[
 ('1','Treat this as benchmarking, not live alerts','Dataset has no event timestamps/IPs and is historical.','Document scope on every dashboard and avoid operational SLA language.','All public project outputs prominently state dataset limitations.'),
 ('2','Prioritize attack recall alongside precision',f"Held-out recall={metrics['attack_recall']:.3f}; false negatives={fn:,}.",'Review false-negative cases by family; set thresholds using validated use-case costs.','Track attack recall, miss rate, and precision together.'),
 ('3','Investigate false positives before escalation',f"Held-out false positives={fp:,}; FPR={metrics['false_positive_rate']:.3f}.",'Review benign records flagged by baseline and design analyst-feedback loop.','Track false positive rate and precision.'),
 ('4','Evaluate unseen attack types separately',f"{len(unseen_labels)} attack labels are absent in train but present in test ({len(unseen):,} records).",'Publish model error breakdown by unseen label; avoid a single aggregate metric.','Track recall on previously unseen labels.'),
 ('5','Improve schema and data quality governance',f"Training missing cells={int(audit.loc[audit.split=='train','missing_cells'].iloc[0])}; test missing cells={int(audit.loc[audit.split=='test','missing_cells'].iloc[0])}.",'Validate 43-column input shape, missing values, categorical drift and duplicates.','Failed schema checks=0; audit outputs retained.'),
 ('6','Use SQL segmentation for analyst-style review',f"Test split includes {int(test.attack_type.nunique())} distinct labels across {int(test.attack_family.nunique())} family classes.",'Maintain family, service and protocol breakdowns to detect benchmark blind spots.','All grouped totals reconcile to source partition.'),
 ('7','Require modern and time-aware validation before deployment','NSL-KDD is an older, non-time-stamped academic dataset.','Assess modern, licensed traffic data with legitimate collection, drift testing and security review before production use.','No production deployment claim without independent modern validation.')]
rec_df=pd.DataFrame(recommendations,columns=['priority','recommendation','evidence','action','success_measure'])
rec_df.to_csv(OUTPUTS/'recommendations.csv',index=False)

def md_table(df):
    df=df.copy();df=df.astype(str).replace({'nan':'—'}); return '| '+' | '.join(df.columns)+' |\n'+'| '+' | '.join(['---']*len(df.columns))+' |\n'+'\n'.join('| '+' | '.join(row.replace('|','/').replace('\n',' ') for row in rows)+' |' for rows in df.values)

readme=f"""# Cybersecurity Network Intrusion & SOC Analytics — Project 11

## Executive summary
Defensive analysis of the **historical NSL-KDD academic intrusion-detection benchmark**, using Python, SQLite SQL, quality audits, a held-out ML classification baseline, and a SOC-style static dashboard. **This is NOT a live security monitoring system** and the record shares are **not real-world incident prevalence**.

## Source and provenance
- Dataset: [NSL-KDD on Kaggle]({SOURCE_URL}).
- Required local downloads: `KDDTrain+.txt`, `KDDTest+.txt` (43 columns, no header). ZIP input supported.
- Academic split: training and held-out test. No event timestamp, real IP address, or real endpoint identifiers available in these files.
- Third-party input files are not redistributed in this project ZIP. Review upstream licensing/terms before publishing data separately.

## Methods
1. Discover and read full `.txt` files or Kaggle ZIP; assert exact 43-column schema.
2. Audit completeness, duplicate rows, class distributions and labels.
3. Aggregate service, protocol and attack family patterns in **SQLite**.
4. Detect attack types appearing only in the official test split.
5. Train a **RandomForestClassifier** on 41 feature columns (exclude labels, difficulty, split); fit preprocessing only on training data, assess untouched test data.
6. Reconcile SQL aggregates and confusion matrix to held-out split.
7. Generate a static PNG dashboard, evidence-based recommendations and machine-readable CSV/JSON artifacts.

## Audited data facts
{md_table(audit)}

### Attack family breakdown — sample composition only
{md_table(family_counts)}

### Baseline holdout evaluation
- Attack precision: **{metrics['attack_precision']:.4f}**
- Attack recall: **{metrics['attack_recall']:.4f}**
- Attack F1: **{metrics['attack_f1']:.4f}**
- False positives: **{fp:,}**, false negatives: **{fn:,}**
- Unseen attack-type labels in test: **{len(unseen_labels)}** ({len(unseen):,} test records)
- Evaluation strictly uses canonical train/test partitions; no randomized mixing across the two splits.
- Caveat: ordinal encodings for categorical feature values are a simple baseline, not an optimized production pipeline.

## Visualization
![Static SOC-style dashboard](figures/soc_executive_dashboard.png)

## Seven data-grounded recommendations
{md_table(rec_df[['priority','recommendation','evidence','action','success_measure']])}

## What this analysis cannot claim
- No hourly/daily anomaly trend: input has no timestamps.
- No real-time threats or current alerts: historical benchmark only.
- No affected customer IPs, devices, organizations or geography.
- No modern SOC production accuracy, impact reduction or live incident response effectiveness.
- The model predicts historical *normal vs attack*; attack-family analysis uses ground-truth dataset labels.

## Reproduce in Google Colab
1. Open `Project_11_Cybersecurity_NSLKDD_SOC_Analytics.ipynb` in Colab.
2. Upload `KDDTrain+.txt` and `KDDTest+.txt`, or the Kaggle ZIP, when Cell 1 prompts you.
3. Click **Runtime → Run all**; Cell 8 packages all generated artifacts into a ZIP.
4. Upload the saved notebook, README, requirements, charts, aggregated outputs and project ZIP contents to GitHub. **Do not upload source records unless permitted by their license.**
5. Verify the GitHub repository against generated figures and metrics; stop when complete and consistent.

## Suggested GitHub repository
`{REPO_NAME}`

## Ethics and responsible use
This project evaluates a historical supervised benchmark for defensive training and learning. It does not provide offensive steps, real-world target selection or access to actual environments. Documentation makes synthetic/illustrative triage distinct from real operational alerts.
"""
(PROJECT/'README.md').write_text(readme,encoding='utf-8')
(PROJECT/'requirements.txt').write_text('numpy>=1.24\npandas>=2.0\nmatplotlib>=3.7\nseaborn>=0.12\nscikit-learn>=1.3\n',encoding='utf-8')
(PROJECT/'.gitignore').write_text('*.zip\n__pycache__/\n.ipynb_checkpoints/\nKDDTrain*\nKDDTest*\n*.arff\n*.joblib\n',encoding='utf-8')
(PROJECT/'LICENSE_DATA_NOTICE.md').write_text(f'NSL-KDD source data are NOT included in this repository. Download from {SOURCE_URL} and check upstream terms before redistribution. This notebook uses NSL-KDD only for defensive educational benchmarking.\n',encoding='utf-8')
(PROJECT/'PROJECT_STATUS.md').write_text('STATUS: LOCALLY GENERATED — GITHUB UPLOAD AND LIVE VERIFICATION PENDING. Do not label FINAL PASS until repo contents are checked.\n',encoding='utf-8')
print('README created, seven recommendations saved.')
print(rec_df[['priority','recommendation']].to_string(index=False))

In [ ]:
# CELL 8 — AUTOMATED QA, FILE MANIFEST, FINAL ZIP & COLAB DOWNLOAD
from zipfile import ZipFile, ZIP_DEFLATED
import hashlib
required=['README.md','requirements.txt','.gitignore','LICENSE_DATA_NOTICE.md','PROJECT_STATUS.md','outputs/data_quality_audit.csv','outputs/attack_family_by_split.csv','outputs/protocol_by_split.csv','outputs/test_service_attack_ranking.csv','outputs/test_flag_summary.csv','outputs/test_attack_label_coverage.csv','outputs/illustrative_triage_categories.csv','outputs/limitations_and_evidence.csv','outputs/holdout_model_metrics.json','outputs/holdout_error_breakdown.csv','outputs/feature_importance.csv','outputs/recommendations.csv','figures/soc_executive_dashboard.png']
missing=[f for f in required if not (PROJECT/f).exists()]
if missing: raise AssertionError('Missing mandatory project files: '+str(missing))
assert len(train)>0 and len(test)>0
assert not (PROJECT/'KDDTrain+.txt').exists() and not (PROJECT/'KDDTest+.txt').exists()
assert sum(pd.read_csv(OUTPUTS/'attack_family_by_split.csv').query("split=='test'").records)==len(test)
assert len(pd.read_csv(OUTPUTS/'recommendations.csv'))==7
assert len(COLUMNS)==43 and len(feature_cols)==41
assert all(0<=metrics[k]<=1 for k in ['accuracy','attack_precision','attack_recall','attack_f1'])
assert 'historical' in (PROJECT/'README.md').read_text(encoding='utf-8').lower()
manifest=[]
for path in sorted(PROJECT.rglob('*')):
    if path.is_file():
        manifest.append({'file':str(path.relative_to(PROJECT)),'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
pd.DataFrame(manifest).to_csv(PROJECT/'FILE_MANIFEST.csv',index=False)
zip_path=ROOT/'Project_11_Cybersecurity_NSLKDD_SOC_Analytics_GitHub_Package.zip'
with ZipFile(zip_path,'w',ZIP_DEFLATED) as z:
    for path in sorted(PROJECT.rglob('*')):
        if path.is_file(): z.write(path,arcname=f'{REPO_NAME}/{path.relative_to(PROJECT)}')
with ZipFile(zip_path) as z:
    assert z.testzip() is None
    names=z.namelist()
    assert len(names)>=len(required)+1
    print('Verified ZIP members:',len(names))
print('PASS — local data aggregates reconcile; ZIP integrity verified.')
print('GitHub upload + live verification remain REQUIRED before project FINAL PASS.')
print('ZIP:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Outside Colab: download ZIP from the displayed path.')